# 플래너 챗봇 버그 수정 스모크 (2026-06-24)

실제 RunPod planner 엔드포인트로 두 결함의 수정을 end-to-end 검증한다.

- **Bug 1** — 크롤/학습에 없는 목표(예: 철인 삼종 경기)에서 관계없는 데이터(시험 task 누수) + '조사'만 생성.
  - 수정: `_DEADLINE_SENSITIVE_WORDS` 에 경기/대회/시합 추가(날짜 되묻기), plan 프롬프트 안티-오염, critic 의 결정적 시험-누수 가드.
  - ⚠️ 근본 해결은 SFT(스펙 Phase 5). 여기 수정은 최악 증상 차단 + 날짜 되묻기.
- **Bug 2** — '8월 8일까지' 라면서 plan 이 7월 7일에서 끊김.
  - 수정: deadline 있으면 코드가 날짜를 소유 — 마일스톤을 today~deadline 전체에 펼치고 마지막날=deadline 보장.

유료 호출 주의. `.env` 의 `RUNPOD_PLANNER_ENDPOINT_URL` / `RUNPOD_API_KEY` 필요.

In [ ]:
import os, sys
from datetime import date, datetime, timedelta
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# .env 로드 (live_planner_smoke.py 와 동일)
for line in (ROOT / ".env").read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if not line or line.startswith("#") or "=" not in line:
        continue
    k, v = line.split("=", 1)
    os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

url = os.environ.get("RUNPOD_PLANNER_ENDPOINT_URL", "")
key = os.environ.get("RUNPOD_API_KEY", "")
assert url and key, "RUNPOD_PLANNER_ENDPOINT_URL / RUNPOD_API_KEY 가 .env 에 없습니다"
print("env ok:", bool(url), bool(key))

In [ ]:
# (선택) 테스트 엔드포인트 오버라이드 — 프로덕션 .env 를 수정하지 말고 여기서만 바꾼다.
# 새로 학습한 LoRA 를 올린 격리 Pod/엔드포인트 URL 을 넣으면 그쪽으로 e2e 한다.
# 비워두면 프로덕션 엔드포인트로 '읽기 추론'만 — 배포/가중치 무손상.
TEST_ENDPOINT_URL = ""  # 예: "https://api.runpod.ai/v2/<test-endpoint-id>"

if TEST_ENDPOINT_URL:
    url = TEST_ENDPOINT_URL
    print("⚠️  테스트 엔드포인트 사용:", url)
else:
    print("프로덕션 엔드포인트 사용 (읽기 추론만 — 배포/가중치 무손상)")

In [ ]:
from adapters.todo_creation.runpod_llm import RunPodQwenLLM
from agents.todo_creation.planner.pipeline import PlannerPorts, get_debug_state, run
from agents.todo_creation.schemas import PlannerInput
from agents.todo_creation.planner.nodes.critic import _EXAM_LEAK_KEYWORDS
from adapters.todo_creation.qwen_llm import _KOREAN_TITLE_PATTERN
import re

_korean_only = re.compile(_KOREAN_TITLE_PATTERN)  # 한국어-only 제목 검사(외국 문자 탐지)

llm = RunPodQwenLLM(endpoint_url=url, api_key=key, adapter="planner")
ports = PlannerPorts(llm=llm)
TODAY = date.today()
NOW = datetime.now()
print("today =", TODAY)


def show(result):
    """결과를 보기 좋게 출력하고 분석용 dict 를 돌려준다."""
    kind = type(result).__name__
    print(f"[{kind}]")
    if kind == "FollowUpResult":
        print("  되묻는 질문:", result.question)
        print("  missing_aspects:", result.missing_aspects)
        return {"kind": kind, "thread_id": result.thread_id}
    if kind == "OutOfScopeResult":
        print("  out_of_scope:", result.message)
        return {"kind": kind, "thread_id": result.thread_id}
    tasks = list(result.todos) + list(result.calendar_events)
    by_day = {}
    for t in tasks:
        by_day.setdefault(t.due_date, []).append(t)
    print("  summary:", result.summary_text)
    for d in sorted(by_day):
        titles = ", ".join(f"{t.title}(난이도 {getattr(t, 'difficulty', 1)})" for t in by_day[d])
        print(f"  {d}: {titles}")
    return {
        "kind": kind,
        "thread_id": result.thread_id,
        "titles": [t.title for t in tasks],
        "due_dates": sorted(t.due_date for t in tasks),
    }


def foreign_titles(titles):
    """한국어-only 패턴에 안 맞는(외국 문자 섞인) 제목을 돌려준다."""
    return [t for t in titles if not _korean_only.match(t)]

## 시나리오 A — Bug 1: 학습에 없는 목표 (철인 삼종 경기)

기대: 날짜를 모르면 바로 생성하지 않고 **경기 날짜를 되묻는다** (이벤트 단어 인식).

In [ ]:
msg1 = "철인 삼종 경기에 나가고 싶어"
print("사용자:", msg1)
first = await run(
    PlannerInput(user_id="smoke", message=msg1, today=TODAY, thread_id=None),
    ports=ports, now=NOW,
)
info1 = show(first)
print()
print("PASS (날짜 되묻기)" if info1["kind"] == "FollowUpResult" else "REVIEW: 되묻지 않고 바로 진행됨")

In [ ]:
# 경기 날짜 제공 → 최종 계획 생성. 기대: 시험 누수 없음 + 외국 문자 없음 + 마지막날 == 8/8
deadline = date(TODAY.year if TODAY <= date(TODAY.year, 8, 8) else TODAY.year + 1, 8, 8)
msg2 = f"{deadline.month}월 {deadline.day}일에 경기가 예정되어 있어요"
print("사용자:", msg2)
final = await run(
    PlannerInput(user_id="smoke", message=msg2, today=TODAY, thread_id=first.thread_id),
    ports=ports, now=NOW,
)
info2 = show(final)

print("\n--- 검증 ---")
if info2["kind"] == "CandidatesResult":
    leaked = [t for t in info2["titles"] if any(k in t for k in _EXAM_LEAK_KEYWORDS)]
    foreign = foreign_titles(info2["titles"])
    last_day = info2["due_dates"][-1] if info2["due_dates"] else None
    print("Bug1 시험-누수:", "PASS (없음)" if not leaked else f"FAIL {leaked}")
    print("외국 문자 제목:", "PASS (없음)" if not foreign else f"FAIL {foreign}  (워커 재배포 필요?)")
    print("Bug2 마지막날==deadline:", "PASS" if last_day == deadline else f"FAIL last={last_day} deadline={deadline}")
else:
    print("REVIEW: 계획이 생성되지 않음 ->", info2["kind"])

## 시나리오 B — Bug 2 직접: 날짜를 한 번에 제공

기대: 한 턴에 목표+마감일을 주면 plan 의 마지막날이 마감일까지 도달(7/7 에서 끊기지 않음).

In [ ]:
deadline_b = TODAY + timedelta(days=45)
msg = f"{deadline_b.month}월 {deadline_b.day}일 마라톤 대회까지 준비 계획 짜줘"
print("사용자:", msg, "(deadline =", deadline_b, ")")
res = await run(
    PlannerInput(user_id="smokeB", message=msg, today=TODAY, thread_id=None),
    ports=ports, now=NOW,
)
infoB = show(res)

print("\n--- 검증 ---")
if infoB["kind"] == "CandidatesResult" and infoB["due_dates"]:
    last_day = infoB["due_dates"][-1]
    span_days = (last_day - TODAY).days
    print("Bug2 마지막날==deadline:", "PASS" if last_day == deadline_b else f"FAIL last={last_day} deadline={deadline_b}")
    print(f"  plan span = {span_days}일 (이전엔 ~7일에서 끊김)")
elif infoB["kind"] == "FollowUpResult":
    print("되묻기 발생 — 같은 thread 로 한 번 더 답하면 계획이 나옵니다.")
else:
    print("REVIEW:", infoB["kind"])